# ***`FINE TUNING FOR POSITIVE/NEGATIVE REVIEWS`***

In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
import numpy as np, evaluate
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)


In [ ]:
# 1. Data (small subsets so it's fast)
ds = load_dataset("stanfordnlp/imdb")
train = ds["train"].shuffle(seed=42).select(range(5000))
test  = ds["test"].shuffle(seed=42).select(range(1000))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
model_id = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_id)
def prep(batch):
    return tok(batch["text"], truncation=True, max_length=256)
train = train.map(prep, batched=True)
test  = test.map(prep, batched=True)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# 3. Model (2 labels: 0 = negative, 1 = positive)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
acc = evaluate.load("accuracy")
def metrics(p):
    return acc.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)


In [ ]:
# 5. Train
args = TrainingArguments(
    output_dir="sentiment-model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    fp16=True,                 # T4 supports fp16
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=test,
                  processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)
before = trainer.evaluate()
print("BEFORE fine-tuning:", before)

trainer.train()

after = trainer.evaluate()
print("AFTER fine-tuning:", after)

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.692563,0,0.518000


BEFORE fine-tuning: {'eval_loss': 0.6925634741783142, 'eval_accuracy': 0.518}


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.428170,0.827000
2,0.324928,0.308995,0.885000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.324928,0.308995,2,0.885000


AFTER fine-tuning: {'eval_loss': 0.30899494886398315, 'eval_accuracy': 0.885}


In [ ]:
print(f"Accuracy before: {before['eval_accuracy']:.3f}")
print(f"Accuracy after:  {after['eval_accuracy']:.3f}")
print(f"Improvement:     {(after['eval_accuracy'] - before['eval_accuracy']) * 100:.1f} points")

Accuracy before: 0.518
Accuracy after:  0.885
Improvement:     36.7 points


In [ ]:

# 6. Try it
from transformers import pipeline
clf = pipeline("sentiment-analysis", model=model, tokenizer=tok, device=0)
print(clf("This movie was a complete waste of time."))
print(clf("Absolutely loved it, great acting!"))

[{'label': 'LABEL_1', 'score': 0.5459204316139221}]
[{'label': 'LABEL_1', 'score': 0.545931339263916}]


In [ ]:
from transformers import pipeline
fill = pipeline("fill-mask", model="distilbert-base-uncased")
print(fill("The movie was really [MASK]."))
print(fill("Spider-Man got his powers from a [MASK] spider."))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[{'score': 0.06833802163600922, 'token': 2204, 'token_str': 'good', 'sequence': 'the movie was really good.'}, {'score': 0.060588203370571136, 'token': 6057, 'token_str': 'funny', 'sequence': 'the movie was really funny.'}, {'score': 0.053074512630701065, 'token': 2919, 'token_str': 'bad', 'sequence': 'the movie was really bad.'}, {'score': 0.04624665156006813, 'token': 12459, 'token_str': 'scary', 'sequence': 'the movie was really scary.'}, {'score': 0.0424809530377388, 'token': 14036, 'token_str': 'entertaining', 'sequence': 'the movie was really entertaining.'}]
[{'score': 0.08145581185817719, 'token': 5016, 'token_str': 'giant', 'sequence': 'spider - man got his powers from a giant spider.'}, {'score': 0.025478100404143333, 'token': 2613, 'token_str': 'real', 'sequence': 'spider - man got his powers from a real spider.'}, {'score': 0.018918229267001152, 'token': 3909, 'token_str': 'flying', 'sequence': 'spider - man got his powers from a flying spider.'}, {'score': 0.01837177015841

In [ ]:
print(fill("The core part of any Operating System is [MASK]."))

[{'score': 0.10619623214006424, 'token': 11603, 'token_str': 'linux', 'sequence': 'the core part of any operating system is linux.'}, {'score': 0.06765387952327728, 'token': 19998, 'token_str': 'unix', 'sequence': 'the core part of any operating system is unix.'}, {'score': 0.04879939556121826, 'token': 4007, 'token_str': 'software', 'sequence': 'the core part of any operating system is software.'}, {'score': 0.03263161703944206, 'token': 9262, 'token_str': 'java', 'sequence': 'the core part of any operating system is java.'}, {'score': 0.03095654770731926, 'token': 8051, 'token_str': 'hardware', 'sequence': 'the core part of any operating system is hardware.'}]


### ***`FINE TUNING FOR OPERATING SYSTEMS SUBJECT`***

In [ ]:
#scraping from web
import requests

titles = ["Operating system", "Kernel (operating system)", "Process (computing)", "Thread (computing)",
          "Scheduling (computing)", "Deadlock (computer science)", "Virtual memory", "Paging",
          "Memory management", "Semaphore (programming)", "Mutual exclusion", "Race condition",
          "File system", "Inode", "Device driver", "Interrupt", "System call", "Context switch",
          "Process management (computing)", "Round-robin scheduling", "Page replacement algorithm",
          "Segmentation (memory)", "Multitasking", "Monolithic kernel", "Microkernel", "Linux kernel",
          "Unix", "Shell (computing)", "Booting", "Thrashing (computer science)",
          "Inter-process communication", "Critical section", "Mutex", "Cache (computing)",
          "Input/output", "Real-time operating system", "Virtual machine", "Hypervisor",
          "Process control block", "Starvation (computer science)"]

lines = []
# Using a more descriptive, compliant User-Agent as per MediaWiki API guidelines
headers = {"User-Agent": "OS-Finetuning-Colab-Demo/1.0 (contact@example.com)"}

for t in titles:
    r = requests.get("https://en.wikipedia.org/w/api.php",
                     params={"action": "query", "prop": "extracts", "explaintext": 1,
                             "redirects": 1, "titles": t, "format": "json"},
                     headers=headers)
    if r.status_code == 200:
        try:
            data = r.json()
            for page in data["query"]["pages"].values():
                lines += [p for p in page.get("extract", "").split("\n") if len(p) > 50]
        except ValueError:
            print(f"Failed to parse JSON for title: {t}")
    else:
        print(f"Request failed for title {t} with status code {r.status_code}")

open("os.txt", "w").write("\n".join(lines))
print(len(lines), "paragraphs saved")

2249 paragraphs saved


In [ ]:
#if scarping from web
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")

ds = load_dataset("text", data_files="os.txt")["train"].train_test_split(test_size=0.1, seed=42)
ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=128, return_overflowing_tokens=True),
            batched=True, remove_columns=["text"])

# ***`Providing own text`***
Rules for any text file
* One paragraph per line, with no blank lines.
* Plain text only (.txt), so convert PDFs or slides to text first.
* Avoid very short lines such as headings or bullet fragments, since they add little to learn from.

In [1]:
text="""An operating system is system software that manages computer hardware and software resources and provides common services for application programs. It acts as an intermediary between the user and the hardware, hiding the complexity of the machine behind a simpler interface. The main goals of an operating system are to execute programs, make the computer convenient to use, and use the hardware efficiently.
The kernel is the core component of an operating system and always resides in main memory. It has complete control over everything in the system, including the cpu, memory, and input and output devices. The kernel runs in privileged mode, also called kernel mode, while ordinary applications run in user mode with restricted access to hardware.
A system call is the programmatic way in which a user program requests a service from the kernel. When a program needs to read a file, create a process, or allocate memory, it issues a system call, which triggers a trap and switches the cpu from user mode to kernel mode. After the kernel finishes the request, control returns to the user program.
A process is a program in execution. Each process has its own address space, which contains the program code, the data section, the heap for dynamic memory, and the stack for function calls and local variables. The operating system keeps track of every process using a data structure called the process control block.
The process control block stores the information the operating system needs to manage a process. It contains the process identifier, the process state, the program counter, the values of the cpu registers, scheduling information, memory management information, and the list of open files. When the cpu switches from one process to another, the state of the old process is saved in its process control block.
A process moves through several states during its lifetime. It is new when it is being created, ready when it is waiting to be assigned to the cpu, running when its instructions are being executed, waiting when it is blocked for an input and output event, and terminated when it has finished execution. The scheduler decides which ready process runs next.
A context switch occurs when the cpu changes from executing one process to executing another. The kernel saves the state of the current process into its process control block and loads the saved state of the next process. Context switching is pure overhead because the system does no useful work while switching, so its cost affects overall performance.
A thread is the smallest unit of execution within a process. Threads of the same process share the code section, the data section, and open files, but each thread has its own program counter, register set, and stack. Multithreading allows a program to perform several tasks concurrently and makes better use of multicore processors.
Cpu scheduling is the basis of multiprogrammed operating systems. By switching the cpu among processes, the operating system can make the computer more productive. The scheduler selects a process from the ready queue and the dispatcher gives it control of the cpu, and the time the dispatcher takes to stop one process and start another is called dispatch latency.
Common scheduling criteria include cpu utilization, throughput, turnaround time, waiting time, and response time. Turnaround time is the total time from submission of a process to its completion, while waiting time is the total time a process spends in the ready queue. A good scheduling algorithm tries to maximize utilization and throughput while minimizing waiting time and response time.
First come first served is the simplest scheduling algorithm, in which the process that requests the cpu first is allocated the cpu first. It is implemented with a fifo queue. However, it can cause the convoy effect, where many short processes wait behind one long process, which increases the average waiting time.
Shortest job first scheduling assigns the cpu to the process with the smallest next cpu burst. It is provably optimal because it gives the minimum average waiting time for a given set of processes. The difficulty is knowing the length of the next cpu burst, which is usually predicted from the lengths of previous bursts using exponential averaging.
Round robin scheduling is designed for time sharing systems. Each process is given a small unit of cpu time called a time quantum, and when the quantum expires the process is preempted and added to the end of the ready queue. If the time quantum is too large, round robin behaves like first come first served, and if it is too small, the overhead of context switching becomes excessive.
Priority scheduling assigns a priority to each process and the cpu is allocated to the process with the highest priority. A major problem is starvation, where low priority processes may wait indefinitely. A common solution is aging, which gradually increases the priority of processes that have been waiting in the system for a long time.
Multilevel queue scheduling partitions the ready queue into several separate queues, such as a foreground queue for interactive processes and a background queue for batch processes. Each queue has its own scheduling algorithm. In multilevel feedback queue scheduling, a process can move between queues, which allows the scheduler to separate processes according to the characteristics of their cpu bursts.
Process synchronization is needed because concurrent processes that share data can produce inconsistent results. A race condition occurs when the outcome of execution depends on the particular order in which the accesses to shared data take place. To avoid race conditions, only one process at a time should be allowed to manipulate the shared variable.
The critical section is a segment of code in which a process accesses shared resources such as variables, tables, or files. A solution to the critical section problem must satisfy three requirements: mutual exclusion, progress, and bounded waiting. Mutual exclusion means that if one process is executing in its critical section, no other process can execute in theirs.
A semaphore is an integer variable that is accessed only through two atomic operations called wait and signal. A binary semaphore can take only the values zero and one and is used for mutual exclusion, while a counting semaphore can take any non negative value and is used to control access to a resource with a finite number of instances. Incorrect use of semaphores can lead to deadlock.
A mutex lock is a simple synchronization tool in which a process must acquire the lock before entering its critical section and release it on exit. If the lock is unavailable, the calling process blocks or spins in a busy loop. A lock that uses busy waiting is called a spinlock, and it is useful on multiprocessor systems when locks are held for very short periods.
The producer consumer problem is a classic synchronization problem in which a producer process generates items and places them in a bounded buffer while a consumer process removes items from the buffer. The producer must wait when the buffer is full and the consumer must wait when it is empty. Semaphores can be used to solve this problem correctly.
The dining philosophers problem illustrates the difficulty of allocating shared resources among processes without causing deadlock or starvation. Five philosophers sit around a table with one chopstick between each pair, and a philosopher needs both adjacent chopsticks to eat. If every philosopher picks up the left chopstick at the same time, a deadlock occurs.
A deadlock is a situation in which a set of processes are blocked because each process holds a resource and waits for another resource held by some other process in the set. Four conditions must hold simultaneously for a deadlock to occur: mutual exclusion, hold and wait, no preemption, and circular wait. Removing any one of these conditions prevents deadlock.
There are several methods for handling deadlocks. The system can use deadlock prevention or avoidance to ensure that it never enters a deadlocked state, it can allow deadlocks to occur and then detect and recover from them, or it can ignore the problem altogether, which is the approach taken by most general purpose operating systems.
The bankers algorithm is a deadlock avoidance algorithm. Before granting a resource request, it checks whether the resulting allocation leaves the system in a safe state, meaning there exists a sequence in which all processes can finish. If granting the request would lead to an unsafe state, the process must wait until its request can be granted safely.
Main memory is a large array of bytes in which programs must be loaded to be executed. Memory management is the function of the operating system that keeps track of which parts of memory are in use and by whom, decides which processes to load when space becomes available, and allocates and deallocates memory as needed.
Contiguous memory allocation assigns each process a single continuous section of memory. As processes are loaded and removed, free memory is broken into many small holes, a problem known as fragmentation. External fragmentation occurs when there is enough total free space but it is not contiguous, while internal fragmentation occurs when a process is given more memory than it requested.
Compaction is a technique for reducing external fragmentation by shifting allocated memory so that all free space is gathered into one large block. Common strategies for choosing a free hole for a process include first fit, best fit, and worst fit. First fit and best fit are generally better than worst fit in terms of speed and storage utilization.
Paging is a memory management scheme that eliminates the need for contiguous allocation of physical memory. Physical memory is divided into fixed size blocks called frames, and logical memory is divided into blocks of the same size called pages. The page table maps each logical page number to a physical frame number, and paging avoids external fragmentation.
A logical address generated by the cpu is divided into a page number and a page offset. The page number is used as an index into the page table to find the frame, and the offset is combined with the frame base address to form the physical address. The memory management unit performs this translation in hardware.
The translation lookaside buffer is a small, fast hardware cache that stores recent page table entries. When the cpu generates a logical address, the tlb is searched first, and if the page number is found, which is called a tlb hit, the frame number is available immediately. On a tlb miss, the page table in memory must be consulted, which takes longer.
Segmentation is a memory management scheme that supports the programmer view of memory. A program is a collection of segments such as the main function, procedures, the stack, and the symbol table. A logical address consists of a segment number and an offset, and the segment table maps each segment to its base address and limit.
Virtual memory is a technique that allows the execution of processes that are not completely in main memory. It separates logical memory from physical memory, so a program can be larger than the available physical memory. Virtual memory is commonly implemented using demand paging, in which pages are loaded only when they are needed.
A page fault occurs when a process tries to access a page that is not currently in main memory. The operating system handles the page fault by finding the page on the backing store, locating a free frame, reading the page into the frame, updating the page table, and restarting the instruction that caused the fault.
When there is no free frame available, the operating system must use a page replacement algorithm to select a victim page to remove from memory. The first in first out algorithm replaces the oldest page, the optimal algorithm replaces the page that will not be used for the longest time, and the least recently used algorithm replaces the page that has not been used for the longest period.
Belady's anomaly is the surprising phenomenon in which increasing the number of page frames results in more page faults for certain reference strings. It occurs with the first in first out replacement algorithm but not with stack algorithms such as least recently used. The optimal algorithm has the lowest page fault rate but cannot be implemented because it requires knowledge of future references.
Thrashing occurs when a process spends more time paging than executing. It happens when the system does not give a process enough frames, so the process continually faults, replacing pages that it needs again immediately. The working set model and page fault frequency control are used to prevent thrashing.
A file is a named collection of related information that is recorded on secondary storage. The operating system provides a uniform logical view of storage through the file abstraction. File attributes include the name, type, location, size, protection information, and timestamps, and common file operations include create, open, read, write, seek, and delete.
The file system organizes files into directories. A directory structure may be single level, two level, or a tree, and modern systems use tree structured directories with absolute and relative path names. Some systems also allow acyclic graph directories that support shared files through links.
File allocation methods determine how disk blocks are assigned to files. Contiguous allocation stores each file in consecutive blocks and gives fast access but suffers from external fragmentation. Linked allocation stores each file as a linked list of blocks, and indexed allocation uses an index block that holds pointers to all the blocks of the file.
In unix style file systems, each file is described by an inode, which stores metadata such as the owner, permissions, timestamps, and pointers to the data blocks. The file name is stored in the directory entry, which maps the name to an inode number. The inode contains direct pointers, a single indirect pointer, a double indirect pointer, and a triple indirect pointer.
Disk scheduling algorithms decide the order in which disk requests are serviced in order to reduce seek time. The fcfs algorithm services requests in arrival order, the sstf algorithm picks the request closest to the current head position, and the scan algorithm, also known as the elevator algorithm, moves the head across the disk servicing requests in one direction before reversing.
An interrupt is a signal sent to the cpu by hardware or software indicating that an event needs immediate attention. When an interrupt occurs, the cpu stops its current work, saves its state, and executes an interrupt service routine. After the routine completes, the cpu resumes the interrupted program. Interrupts allow devices to signal completion of input and output operations without constant polling.
A device driver is a piece of software that allows the operating system to communicate with a particular hardware device. It translates general commands from the kernel into device specific operations. Input and output can be performed using programmed input and output, interrupt driven input and output, or direct memory access, which transfers data between a device and memory without involving the cpu for every byte.
A shell is a program that provides an interface to the services of the operating system. A command line shell accepts text commands from the user, interprets them, and asks the kernel to execute them, while a graphical shell provides windows and icons. Common unix shells include bash and zsh.
In a monolithic kernel, all operating system services such as scheduling, file systems, memory management, and device drivers run in kernel space as a single large program. This design is fast because components communicate through direct function calls. A microkernel instead keeps only the essential functions in the kernel and runs other services as user level processes that communicate through message passing, which improves reliability and modularity.
Inter process communication allows cooperating processes to exchange data and synchronize their actions. The two fundamental models are shared memory and message passing. In shared memory, processes read and write to a common region of memory, while message passing uses send and receive operations, and is useful for exchanging smaller amounts of data between processes on different machines.
Common inter process communication mechanisms include pipes, named pipes, message queues, shared memory segments, sockets, and signals. A pipe provides a unidirectional channel in which the output of one process becomes the input of another. Sockets allow processes to communicate across a network using protocols such as tcp and udp.
A real time operating system is designed to process data and respond to events within strict time constraints. In a hard real time system, missing a deadline is a system failure, as in an airbag controller, while in a soft real time system missing an occasional deadline only degrades quality. Real time schedulers commonly use algorithms such as rate monotonic scheduling and earliest deadline first.
A virtual machine is a software emulation of a computer system that runs its own operating system and applications. A hypervisor is the software layer that creates and manages virtual machines and allocates the physical resources of the host among them. A type one hypervisor runs directly on the hardware, while a type two hypervisor runs on top of a host operating system.
When a computer is powered on, the booting process begins. The firmware, such as the bios or uefi, performs a power on self test and then locates the bootloader on a storage device. The bootloader loads the kernel into main memory and transfers control to it, and the kernel then initializes the hardware and starts the first user process.
Operating system protection ensures that processes cannot interfere with each other or with the operating system itself. Hardware support includes dual mode operation, memory protection using base and limit registers, and a timer that prevents a process from holding the cpu forever. Security mechanisms add authentication, access control lists, and permissions on files and resources.
Multitasking allows several processes to share the cpu by rapidly switching between them so that users perceive them as running at the same time. In a multiprogramming system, several jobs are kept in memory so the cpu always has something to execute, which increases utilization. A multiprocessor system uses more than one cpu and can execute several processes truly in parallel."""

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")

# Split the text string by newlines into a list of sentences/paragraphs
lines_list = [line.strip() for line in text.split("\n") if line.strip()]

# Load the dataset directly from the Python list in memory
ds = Dataset.from_dict({"text": lines_list}).train_test_split(test_size=0.1, seed=42)

ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=128, return_overflowing_tokens=True),
            batched=True, remove_columns=["text"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Map:   0%|          | 0/46 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [3]:
from transformers import pipeline

def test():
    fill = pipeline("fill-mask", model=model, tokenizer=tok, device=0)
    for s in ["The [MASK] schedules processes on the cpu.",
              "A page [MASK] occurs when a process accesses memory that is not loaded.",
              "Deadlock happens when two processes wait for each other's [MASK].",
              "The operating system uses a [MASK] to switch between processes."]:
        print(s, "→", [r["token_str"] for r in fill(s)[:5]])

test()   # BEFORE training: save this output

The [MASK] schedules processes on the cpu. → ['compiler', 'processor', 'cpu', 'algorithm', 'kernel']
A page [MASK] occurs when a process accesses memory that is not loaded. → ['load', 'closure', 'cache', 'error', 'crash']
Deadlock happens when two processes wait for each other's [MASK]. → ['execution', 'input', 'inputs', 'arrival', 'response']
The operating system uses a [MASK] to switch between processes. → ['switch', 'thread', 'processor', 'computer', 'kernel']


In [4]:
import math
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="os-model", num_train_epochs=10, per_device_train_batch_size=8,
                           learning_rate=5e-5, eval_strategy="epoch", fp16=True, report_to="none"),
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=DataCollatorForLanguageModeling(tok, mlm_probability=0.15),
    processing_class=tok)

before = trainer.evaluate()["eval_loss"]
trainer.train()
after = trainer.evaluate()["eval_loss"]
print(f"Perplexity before: {math.exp(before):.1f} → after: {math.exp(after):.1f}")

Training Loss,Validation Loss,Epoch
No log,2.951590,0


Epoch,Training Loss,Validation Loss
1,No log,3.027827
2,No log,2.326888
3,No log,2.201878
4,No log,1.617452
5,No log,1.889837
6,No log,2.318975
7,No log,1.952772
8,No log,1.857740
9,No log,2.386499
10,No log,1.584035


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
No log,1.605766,10


Perplexity before: 19.1 → after: 5.0


In [5]:
test()   # AFTER training: compare with the earlier output

The [MASK] schedules processes on the cpu. → ['kernel', 'algorithm', 'software', 'cpu', 'compiler']
A page [MASK] occurs when a process accesses memory that is not loaded. → ['loss', 'problem', 'interrupt', 'fault', 'closure']
Deadlock happens when two processes wait for each other's [MASK]. → ['execution', 'input', 'return', 'arrival', 'response']
The operating system uses a [MASK] to switch between processes. → ['kernel', 'computer', 'program', 'device', 'timer']


In [6]:
trainer.save_model("os-model-final")
tok.save_pretrained("os-model-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('os-model-final/tokenizer_config.json', 'os-model-final/tokenizer.json')

In [ ]:
!zip -r os-model-final.zip os-model-final

  adding: os-model-final/ (stored 0%)
  adding: os-model-final/config.json (deflated 47%)
  adding: os-model-final/model.safetensors (deflated 8%)
  adding: os-model-final/training_args.bin (deflated 54%)
  adding: os-model-final/tokenizer.json (deflated 71%)
  adding: os-model-final/tokenizer_config.json (deflated 43%)


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
repo = "aditya500/distilbert-os-fill-mask"
model.push_to_hub(repo, private=False)
tok.push_to_hub(repo, private=False)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sgedg0n/model.safetensors:   0%|          |  575kB /  268MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/aditya500/distilbert-os-fill-mask/commit/4f187e7c8693bab3dd05f06e98687993f7b848db', commit_message='Upload tokenizer', commit_description='', oid='4f187e7c8693bab3dd05f06e98687993f7b848db', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aditya500/distilbert-os-fill-mask', endpoint='https://huggingface.co', repo_type='model', repo_id='aditya500/distilbert-os-fill-mask'), pr_revision=None, pr_num=None)

In [ ]:
import math
from huggingface_hub import HfApi

card = f"""---
license: apache-2.0
base_model: distilbert-base-uncased
pipeline_tag: fill-mask
tags: [operating-systems]
---
# DistilBERT fine-tuned on operating systems text

`distilbert-base-uncased` continued-pretrained with masked language modeling on text about
operating systems (processes, scheduling, deadlocks, memory management, file systems).

| | Perplexity (lower is better) |
|---|---|
| Before fine-tuning | {math.exp(before):.1f} |
| After fine-tuning | {math.exp(after):.1f} |

Trained on a small corpus as a learning project, so expect modest gains.
Use it with: `pipeline("fill-mask", model="{repo}")`
"""
HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)

CommitInfo(commit_url='https://huggingface.co/aditya500/distilbert-os-fill-mask/commit/08e35f99bc21a37f52cca11078be918a67b0f2ed', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='08e35f99bc21a37f52cca11078be918a67b0f2ed', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aditya500/distilbert-os-fill-mask', endpoint='https://huggingface.co', repo_type='model', repo_id='aditya500/distilbert-os-fill-mask'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline
fill = pipeline("fill-mask", model=repo)
print(fill("operating system has a pcb and it is called task [MASK]."))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[{'score': 0.09697950631380081, 'token': 3208, 'token_str': 'manager', 'sequence': 'operating system has a pcb and it is called task manager.'}, {'score': 0.03055988810956478, 'token': 2968, 'token_str': 'management', 'sequence': 'operating system has a pcb and it is called task management.'}, {'score': 0.02749883569777012, 'token': 5549, 'token_str': 'mode', 'sequence': 'operating system has a pcb and it is called task mode.'}, {'score': 0.0259233508259058, 'token': 8241, 'token_str': 'server', 'sequence': 'operating system has a pcb and it is called task server.'}, {'score': 0.025158852338790894, 'token': 2491, 'token_str': 'control', 'sequence': 'operating system has a pcb and it is called task control.'}]


# ***TO GITHUB***


```

